# 7. Extra: Cross-Model Scoring Summary

Load test-set scores from the `scoring` MLflow experiment and build summary pivot tables
comparing all models across all time-series lengths (25, 50, 100).

Run names in the experiment are expected to follow the pattern: `{model}_{n_features}_{run_id}`

In [ ]:
import re

import mlflow
import pandas as pd

from utils.config import load_config

config = load_config()


In [ ]:
def load_scoring_pivot(experiment_name: str, metric: str) -> pd.DataFrame:
    """
    Load a scoring metric from MLflow and return a pivot table:
    rows = model_name, columns = number_of_features.

    Run names are expected to follow the pattern: {model}_{n_features}_{run_id}
    """
    experiment = mlflow.get_experiment_by_name(experiment_name)
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        run_view_type=mlflow.entities.ViewType.ALL,
    )

    pattern = re.compile(r"(.+)_(\d+)_(.+)")
    records = []
    for _, row in runs.iterrows():
        match = pattern.match(row.get("tags.mlflow.runName", ""))
        if not match:
            continue
        value = row.get(f"params.{metric}") or row.get(f"metrics.{metric}")
        if value is None:
            continue
        records.append(
            {
                "model_name": match.group(1),
                "n_features": int(match.group(2)),
                metric: float(value),
            }
        )

    df = pd.DataFrame(records)
    return df.pivot_table(
        index="model_name",
        columns="n_features",
        values=metric,
        aggfunc="min",
    ).sort_values(by=sorted(df["n_features"].unique().tolist()), ascending=False)


In [ ]:
# RMSE comparison: lower is better
rmse_pivot = load_scoring_pivot(config.scoring_experiment_name, "rmse")
print("RMSE by model and time-series length")
rmse_pivot


In [ ]:
# Error std comparison: lower indicates more consistent predictions
std_pivot = load_scoring_pivot(config.scoring_experiment_name, "error_std")
print("Error Std by model and time-series length")
std_pivot
